# Unsupervised endpoint and full-range methods

The two methods are kept explicit: endpoint_validation and full_range.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "phase_transitions").exists():
    ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from phase_transitions.config import TRANSITIONS
from phase_transitions.reporting import write_result
from phase_transitions.unsupervised import (
    EndpointValidationError,
    run_endpoint_clustering,
    run_full_range_clustering,
)

In [ ]:
TRANSITION_NAMES = ["AB", "BCB", "CA"]
MODEL_NAMES = ["kmeans", "gaussian_mixture", "bayesian_gmm", "meanshift", "birch"]
FEATURE_SET_NAMES = ["30", "20", "12"]
METHODS = ["endpoint_validation", "full_range"]
STANDARDIZE = False
TRAIN_STRIDE = None
FULL_STRIDE = None
BLOCK_COUNTS = range(2, 101)
OUTPUT_ROOT = ROOT / "Results" / "REFactored"
RUN = False  # Set True only after reviewing the matrix.

In [ ]:
if RUN:
    for method in METHODS:
        for transition_name in TRANSITION_NAMES:
            config = TRANSITIONS[transition_name]
            for model_name in MODEL_NAMES:
                for feature_set_name in FEATURE_SET_NAMES:
                    common = dict(
                        config=config,
                        model_name=model_name,
                        feature_set_name=feature_set_name,
                        standardize=STANDARDIZE,
                        block_counts=BLOCK_COUNTS,
                    )
                    if method == "endpoint_validation":
                        try:
                            result = run_endpoint_clustering(
                                train_stride=TRAIN_STRIDE,
                                full_stride=FULL_STRIDE,
                                **common,
                            )
                        except EndpointValidationError as exc:
                            print("SKIPPED:", exc)
                            continue
                    else:
                        result = run_full_range_clustering(
                            full_stride=FULL_STRIDE,
                            **common,
                        )
                    summary_path, curve_path = write_result(
                        result,
                        OUTPUT_ROOT / "unsupervised" / method / transition_name,
                    )
                    print(summary_path, curve_path)
                    print("validation:", result.validation_accuracy)
                    print("P=0.5:", result.crossing, "chi peak:", result.chi_peak)
else:
    print("RUN=False: no experiment was executed.")